In [ ]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_DIR = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\VideoMAEv2-master"
os.chdir(REPO_DIR)

print("cwd:", os.getcwd())
print("repo files (sample):", [x for x in os.listdir(".")][:30])

In [ ]:
import subprocess, sys

cmd = [sys.executable, "extract_tad_feature.py", "-h"]
p = subprocess.run(cmd, capture_output=True, text=True)

print("RETURN CODE:", p.returncode)
print("\n--- STDOUT ---\n", p.stdout[:4000])
print("\n--- STDERR ---\n", p.stderr[:4000])

In [ ]:
import timm
import torch

print("timm:", timm.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

model = timm.create_model("vit_base_patch16_224.mae", pretrained=True)

model = model.cuda()
model.eval()

print("Model loaded successfully")

In [ ]:
# ============================================================
# VIDEO → TIME-RESOLVED EMBEDDINGS (T x 768) USING TIMM MAE
# Saves:
#   - <video_stem>__embeddings.npy     (T x 768)
#   - <video_stem>__timestamps.npy     (T,)
#   - video_embeddings_index.csv       (index of outputs)
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import timm

from decord import VideoReader, cpu
import cv2  # used for resizing

# ---------------- USER SETTINGS ----------------
VIDEO_DIR = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\EI Videos"     # <-- change this
OUT_DIR   = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\Features"  # <-- change this

# How densely to sample clips (seconds between sampled frames/embeds)
STRIDE_SEC = .5   # 1s gives decent temporal resolution for peak timing

# MAE input size
IMG_SIZE = 224

# ---------------- SETUP ----------------
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Use the model you already loaded successfully
model = timm.create_model("vit_base_patch16_224.mae", pretrained=True)
model = model.to(device)
model.eval()

# MAE expects normalized ImageNet-like inputs
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_frame(frame_rgb_uint8: np.ndarray) -> torch.Tensor:
    """
    frame_rgb_uint8: H x W x 3 (RGB, uint8)
    returns: 1 x 3 x 224 x 224 float32 normalized tensor
    """
    # Resize to 224x224
    img = cv2.resize(frame_rgb_uint8, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    img = np.transpose(img, (2, 0, 1))  # 3 x H x W
    x = torch.from_numpy(img).unsqueeze(0)  # 1 x 3 x 224 x 224
    return x

@torch.no_grad()
def encode_batch(frames_rgb_uint8: np.ndarray) -> np.ndarray:
    """
    frames_rgb_uint8: N x H x W x 3 (RGB uint8)
    returns: N x 768 embeddings
    """
    # Preprocess batch
    xs = torch.cat([preprocess_frame(fr) for fr in frames_rgb_uint8], dim=0).to(device)  # N x 3 x 224 x 224

    # Forward through MAE backbone:
    # In timm MAE models, `forward_features` typically returns token embeddings.
    # We'll pool tokens to a single vector per frame.
    feats = model.forward_features(xs)  # shape often: N x (num_tokens) x dim  OR N x dim depending on impl

    if feats.ndim == 3:
        # token sequence: average pool tokens
        emb = feats.mean(dim=1)
    else:
        emb = feats

    return emb.detach().cpu().numpy()

def extract_video_embeddings(video_path: str, stride_sec: float = 1.0, batch_size: int = 16):
    vr = VideoReader(video_path, ctx=cpu(0))
    fps = float(vr.get_avg_fps())
    n_frames = len(vr)
    dur_sec = n_frames / fps if fps > 0 else np.nan

    stride_frames = max(1, int(round(stride_sec * fps)))
    frame_idxs = np.arange(0, n_frames, stride_frames, dtype=int)

    timestamps = frame_idxs / fps

    embs = []
    # batch over frame indices
    for i in range(0, len(frame_idxs), batch_size):
        idx_batch = frame_idxs[i:i+batch_size].tolist()
        frames = vr.get_batch(idx_batch).asnumpy()  # N x H x W x 3 (RGB)
        emb = encode_batch(frames)
        embs.append(emb)

    embs = np.vstack(embs)  # T x 768
    return {
        "fps": fps,
        "n_frames": n_frames,
        "dur_sec": dur_sec,
        "stride_sec": stride_sec,
        "frame_idxs": frame_idxs,
        "timestamps": timestamps,
        "embeddings": embs
    }

# ---------------- RUN EXTRACTION ----------------
video_files = []
for ext in ("*.mp4", "*.mov", "*.mkv", "*.avi"):
    video_files.extend(glob.glob(os.path.join(VIDEO_DIR, ext)))

video_files = sorted(video_files)
print("Found videos:", len(video_files))

rows = []

for vp in video_files:
    vp = str(vp)
    stem = Path(vp).stem

    print("\nExtracting:", stem)
    out_emb = os.path.join(OUT_DIR, f"{stem}__embeddings.npy")
    out_ts  = os.path.join(OUT_DIR, f"{stem}__timestamps.npy")

    payload = extract_video_embeddings(vp, stride_sec=STRIDE_SEC, batch_size=16)

    np.save(out_emb, payload["embeddings"])
    np.save(out_ts,  payload["timestamps"])

    # also compute a single pooled embedding for easy regression
    pooled = payload["embeddings"].mean(axis=0)  # 768
    out_pool = os.path.join(OUT_DIR, f"{stem}__pooled.npy")
    np.save(out_pool, pooled)

    rows.append({
        "video_path": vp,
        "video_stem": stem,
        "fps": payload["fps"],
        "n_frames": payload["n_frames"],
        "dur_sec": payload["dur_sec"],
        "stride_sec": payload["stride_sec"],
        "T": payload["embeddings"].shape[0],
        "D": payload["embeddings"].shape[1],
        "embeddings_npy": out_emb,
        "timestamps_npy": out_ts,
        "pooled_npy": out_pool,
    })

index_df = pd.DataFrame(rows)
index_csv = os.path.join(OUT_DIR, "video_embeddings_index.csv")
index_df.to_csv(index_csv, index=False)

print("\nDone.")
print("Index saved to:", index_csv)
index_df.head()

In [ ]:
# ============================================================
# BUILD VIDEO-LEVEL FEATURE TABLE (pooled embeddings)
# Output: video_feat_df  (one row per video, 768 feature cols)
# ============================================================

import numpy as np
import pandas as pd

INDEX_CSV = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\Features\video_embeddings_index.csv"

idx = pd.read_csv(INDEX_CSV)

# Load pooled embedding vectors
feat_rows = []
for _, r in idx.iterrows():
    pooled = np.load(r["pooled_npy"])
    feat_rows.append(pooled)

X = np.vstack(feat_rows)  # n_videos x 768

feat_cols = [f"feat_{i:03d}" for i in range(X.shape[1])]
video_feat_df = pd.concat(
    [idx[["video_stem", "video_path", "dur_sec", "stride_sec", "T", "D"]].reset_index(drop=True),
     pd.DataFrame(X, columns=feat_cols)],
    axis=1
)

print("video_feat_df:", video_feat_df.shape)
video_feat_df.head()

In [ ]:
# ============================================================
# BUILD GROUP-LEVEL MODELING TABLE (28-video set only)
# - Loads VideoMAE pooled embeddings (768-D) from video_embeddings_index.csv
# - Loads mapping: video_mapping.csv (video -> trigger_number)
# - Loads behavioral video-level performance CSV
# - Loads ISC CSV (wide Trigger+ROI cols OR long Subject/Trigger/ROI/ISC)
# - DROPS the 2 extra videos that are not in the 28-video mapping
#
# OUTPUT:
#   - video_feat_df (28 rows)
#   - video_performance_df (28 rows)
#   - video_group_roi_isc_wide (28 rows)
#   - modeling_df (28 rows; merged)
# ============================================================

import os
import re
import numpy as np
import pandas as pd

# ---------------- USER PATHS (EDIT THESE) ----------------
EMB_INDEX_CSV  = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\Features\video_embeddings_index.csv"
VIDEO_MAP_CSV  = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_mapping.csv"
BEHAVIOR_CSV   = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_performance_v1.csv"      # <-- EDIT
ISC_CSV        = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_group_roi_isc_wide_v1.csv"    # <-- EDIT
# ---------------------------------------------------------

def norm_name(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.(mp4|mov|mkv|avi)$", "", s, flags=re.IGNORECASE)
    return s

# ============================================================
# 1) LOAD VIDEO MAPPING (EXPLICIT COLUMN NAMES)
# ============================================================
video_map_df = pd.read_csv(VIDEO_MAP_CSV).copy()
video_map_df = video_map_df.rename(columns={"video": "VideoKey", "trigger_number": "Trigger"})
video_map_df["VideoKey"] = video_map_df["VideoKey"].apply(norm_name)
video_map_df["Trigger"] = pd.to_numeric(video_map_df["Trigger"], errors="coerce")
video_map_df = video_map_df.dropna(subset=["VideoKey", "Trigger"]).drop_duplicates(subset=["VideoKey", "Trigger"])
video_map_df["Trigger"] = video_map_df["Trigger"].astype(int)

print("video_map_df:", video_map_df.shape)
display(video_map_df.head())

# ============================================================
# 2) LOAD VIDEO EMBEDDINGS (POOLED) -> video_feat_df
#    + DROP videos not in mapping (keeps only 28)
# ============================================================
idx = pd.read_csv(EMB_INDEX_CSV).copy()
idx["video_stem"] = idx["video_stem"].apply(norm_name)
idx["VideoKey"] = idx["video_stem"]

# Keep only videos that exist in mapping (this drops the 2 extras)
idx = idx.merge(video_map_df, on="VideoKey", how="inner")

# load pooled vectors for the filtered (28) videos
X = np.vstack([np.load(p) for p in idx["pooled_npy"]])
feat_cols = [f"feat_{i:03d}" for i in range(X.shape[1])]

video_feat_df = pd.concat(
    [
        idx[["VideoKey", "Trigger", "video_path", "video_stem", "dur_sec", "stride_sec", "T", "D"]].reset_index(drop=True),
        pd.DataFrame(X, columns=feat_cols)
    ],
    axis=1
)

print("video_feat_df (filtered to mapping):", video_feat_df.shape)
display(video_feat_df.head())

# ============================================================
# 3) LOAD BEHAVIOR PERFORMANCE -> video_performance_df
# ============================================================
video_performance_df = pd.read_csv(BEHAVIOR_CSV).copy()

if "BaseVideo" in video_performance_df.columns:
    video_performance_df["VideoKey"] = video_performance_df["BaseVideo"].apply(norm_name)
elif "Video" in video_performance_df.columns:
    video_performance_df["VideoKey"] = video_performance_df["Video"].apply(norm_name)
else:
    raise ValueError("Behavior CSV must contain either 'BaseVideo' or 'Video'.")

# attach Trigger via mapping
video_performance_df = video_performance_df.merge(video_map_df, on="VideoKey", how="inner")  # inner keeps only mapped 28

print("video_performance_df:", video_performance_df.shape)
display(video_performance_df.head())

# ============================================================
# 4) LOAD ISC -> video_group_roi_isc_wide
#    Supports:
#      A) already wide: Trigger + ROI columns
#      B) long: Subject, Trigger, ROI, ISC
# ============================================================
isc_raw = pd.read_csv(ISC_CSV).copy()
cols_lower = {c: c.lower() for c in isc_raw.columns}
is_long = ("roi" in cols_lower.values()) and ("isc" in cols_lower.values())

if is_long:
    roi_col  = [c for c in isc_raw.columns if c.lower() == "roi"][0]
    isc_col  = [c for c in isc_raw.columns if c.lower() == "isc"][0]
    trig_col = [c for c in isc_raw.columns if c.lower() == "trigger"][0]

    isc_raw["Trigger"] = pd.to_numeric(isc_raw[trig_col], errors="coerce")
    isc_raw[isc_col] = pd.to_numeric(isc_raw[isc_col], errors="coerce")

    video_group_roi_isc = (
        isc_raw.dropna(subset=["Trigger", roi_col, isc_col])
        .groupby(["Trigger", roi_col], as_index=False)
        .agg(Mean_ISC=(isc_col, "mean"))
        .rename(columns={roi_col: "ROI"})
    )

    video_group_roi_isc_wide = (
        video_group_roi_isc
        .pivot_table(index="Trigger", columns="ROI", values="Mean_ISC", aggfunc="mean")
        .reset_index()
    )
    video_group_roi_isc_wide.columns.name = None

else:
    # assume already wide
    video_group_roi_isc_wide = isc_raw.copy()

    if "Trigger" not in video_group_roi_isc_wide.columns:
        trig_candidates = [c for c in video_group_roi_isc_wide.columns if "trig" in c.lower() or c.lower() == "trigger"]
        if trig_candidates:
            video_group_roi_isc_wide["Trigger"] = pd.to_numeric(video_group_roi_isc_wide[trig_candidates[0]], errors="coerce")
        else:
            raise ValueError("ISC wide CSV must contain a Trigger column (or a trigger-like column).")

# Clean + ensure int Trigger
video_group_roi_isc_wide["Trigger"] = pd.to_numeric(video_group_roi_isc_wide["Trigger"], errors="coerce")
video_group_roi_isc_wide = video_group_roi_isc_wide.dropna(subset=["Trigger"])
video_group_roi_isc_wide["Trigger"] = video_group_roi_isc_wide["Trigger"].astype(int)

# Attach VideoKey via mapping and keep only mapped 28 triggers
video_group_roi_isc_wide = video_group_roi_isc_wide.merge(
    video_map_df[["Trigger", "VideoKey"]],
    on="Trigger",
    how="inner"
)

print("video_group_roi_isc_wide (filtered to mapping):", video_group_roi_isc_wide.shape)
display(video_group_roi_isc_wide.head())

# ============================================================
# 5) FINAL MERGE -> modeling_df (28 rows)
# ============================================================
modeling_df = video_feat_df.copy()

# Merge behavior (one row per video)
beh_drop = ["BaseVideo", "Video"]
beh_merge = video_performance_df.drop(columns=beh_drop, errors="ignore").drop_duplicates(subset=["VideoKey", "Trigger"])
modeling_df = modeling_df.merge(beh_merge, on=["VideoKey", "Trigger"], how="inner", suffixes=("", "_beh"))

# Merge ISC
isc_merge = video_group_roi_isc_wide.drop_duplicates(subset=["VideoKey", "Trigger"])
modeling_df = modeling_df.merge(isc_merge, on=["VideoKey", "Trigger"], how="inner", suffixes=("", "_isc"))

print("\nmodeling_df (28-video set):", modeling_df.shape)

# Diagnostics
if "Avg_Performance_Z" in modeling_df.columns:
    print("Missing behavior rows:", modeling_df["Avg_Performance_Z"].isna().sum(), "/", len(modeling_df))

roi_cols = [c for c in modeling_df.columns if c.startswith("Left ") or c.startswith("Right ")]
if roi_cols:
    print("Videos missing all ROI ISC:", modeling_df[roi_cols].isna().all(axis=1).sum(), "/", len(modeling_df))

display(modeling_df.head(10))

# Save
OUT_MODELING_CSV = os.path.join(os.path.dirname(EMB_INDEX_CSV), "modeling_df_video_level_28only.csv")
modeling_df.to_csv(OUT_MODELING_CSV, index=False)
print("\nSaved:", OUT_MODELING_CSV)

In [ ]:
# ============================================================
# MODEL COMPARISON (Group-level, n=28 videos)
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr
from sklearn.model_selection import LeaveOneOut, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# ---------------- USER PATHS ----------------
EMB_INDEX_CSV  = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\Features\video_embeddings_index.csv"
VIDEO_MAP_CSV  = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_mapping.csv"
BEHAVIOR_CSV   = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_performance_v1.csv"
ISC_CSV        = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Emotion Perception Data\NN Data\video_group_roi_isc_wide_v1.csv"
# -------------------------------------------------------

OUT_DIR = os.path.join(os.path.dirname(EMB_INDEX_CSV), "model_outputs_elasticnet_bayes_across_video_pca")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
N_VIDEO_PCS = 12
ALPHA_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
L1_RATIO_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
INNER_CV_SPLITS = 5
RANDOM_STATE = 42


# ---------------- HELPERS ----------------
def norm_name(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.(mp4|mov|mkv|avi)$", "", s, flags=re.IGNORECASE)
    return s

def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if pd.isna(sd) or sd == 0:
        return s - mu
    return (s - mu) / sd

def orient_embeddings(arr):
   
    arr = np.asarray(arr)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D embeddings array, got shape {arr.shape}")

    # If first dimension looks like features and second looks like time, transpose
    if arr.shape[0] >= arr.shape[1]:
        arr = arr.T

    return arr

def find_behavior_target_col(df):
    
    preferred = [
        "Avg_Performance_Zscored",
        "Avg_Performance_ZScored",
        "avg_performance_zscored",
        "Avg_Performance_Z_scored",
        "Avg_Performance_Zscore",
    ]
    for c in preferred:
        if c in df.columns:
            return c

    # looser fallback
    for c in df.columns:
        cl = c.lower().replace(" ", "").replace("-", "").replace(".", "").replace("__", "_")
        if "avg" in cl and "performance" in cl and "zscored" in cl:
            return c

    raise ValueError(
        f"Could not find Avg_Performance_Zscored-like column in behavior CSV. "
        f"Columns are: {list(df.columns)}"
    )

def build_across_video_pca_features(index_df, n_video_pcs=N_VIDEO_PCS):
    
    if "embeddings_npy" not in index_df.columns:
        raise ValueError("EMB_INDEX_CSV must contain an 'embeddings_npy' column.")

    records = []
    mean_vecs = []

    for _, row in index_df.iterrows():
        video_key = row["VideoKey"]
        trigger = row["Trigger"]
        emb_path = row["embeddings_npy"]

        emb = np.load(emb_path)
        emb = orient_embeddings(emb)  # rows=time/windows, cols=features

        # Collapse each video to a single comparable vector
        mean_vec = np.nanmean(emb, axis=0)

        rec = {"VideoKey": video_key, "Trigger": trigger}
        records.append(rec)
        mean_vecs.append(mean_vec)

    mean_mat = np.vstack(mean_vecs)
    n_features = mean_mat.shape[1]
    mean_cols = [f"MeanEmb_{i:03d}" for i in range(n_features)]

    video_mean_df = pd.concat(
        [
            pd.DataFrame(records),
            pd.DataFrame(mean_mat, columns=mean_cols)
        ],
        axis=1
    )

    # Standardize ACROSS VIDEOS for PCA
    scaler = StandardScaler()
    mean_mat_std = scaler.fit_transform(mean_mat)

    max_pcs = min(n_video_pcs, mean_mat_std.shape[0], mean_mat_std.shape[1])
    pca = PCA(n_components=max_pcs, random_state=RANDOM_STATE)
    pcs = pca.fit_transform(mean_mat_std)

    pc_cols = [f"VideoPC{i+1:02d}" for i in range(max_pcs)]
    video_pc_df = pd.concat(
        [
            video_mean_df[["VideoKey", "Trigger"]].reset_index(drop=True),
            pd.DataFrame(pcs, columns=pc_cols)
        ],
        axis=1
    )

    # If fewer than requested PCs, pad with zeros so downstream code stays stable
    for i in range(max_pcs, n_video_pcs):
        video_pc_df[f"VideoPC{i+1:02d}"] = 0.0

    # Reorder columns
    full_pc_cols = [f"VideoPC{i+1:02d}" for i in range(n_video_pcs)]
    video_pc_df = video_pc_df[["VideoKey", "Trigger"] + full_pc_cols]

    scree_df = pd.DataFrame({
        "PC": [f"PC{i+1:02d}" for i in range(max_pcs)],
        "ExplainedVarianceRatio": pca.explained_variance_ratio_,
        "CumulativeVarianceExplained": np.cumsum(pca.explained_variance_ratio_)
    })

    return video_pc_df, scree_df, video_mean_df

def plot_scree(scree_df, out_path):
    """
    Plot scree plot for across-video PCA.
    """
    x = np.arange(1, len(scree_df) + 1)
    evr = scree_df["ExplainedVarianceRatio"].to_numpy()
    cev = scree_df["CumulativeVarianceExplained"].to_numpy()

    plt.figure(figsize=(10, 6))
    plt.bar(x, evr, alpha=0.7, label="Explained variance ratio")
    plt.plot(x, cev, marker="o", linewidth=2, label="Cumulative variance explained")
    plt.xticks(x, scree_df["PC"].tolist(), rotation=45)
    plt.xlabel("Principal Component")
    plt.ylabel("Variance Explained")
    plt.title("Scree Plot: PCA Across Videos (mean embedding per video)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

def nested_loocv_elastic_net(
    X,
    y,
    alpha_grid=ALPHA_GRID,
    l1_ratio_grid=L1_RATIO_GRID,
    inner_cv_splits=INNER_CV_SPLITS,
    impute_strategy="mean",
    random_state=RANDOM_STATE,
):
    """
    Nested LOOCV Elastic Net:
      - outer loop: leave-one-video-out
      - inner loop: tune alpha and l1_ratio on training data only
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    loo = LeaveOneOut()
    yhat = np.full_like(y, np.nan, dtype=float)
    best_params_records = []

    for fold, (train_idx, test_idx) in enumerate(loo.split(X), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]

        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy=impute_strategy)),
            ("scaler", StandardScaler()),
            ("enet", ElasticNet(max_iter=100000, random_state=random_state))
        ])

        inner_splits = min(inner_cv_splits, len(train_idx))
        if inner_splits < 2:
            raise ValueError("Not enough training samples for inner CV.")

        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

        param_grid = {
            "enet__alpha": alpha_grid,
            "enet__l1_ratio": l1_ratio_grid
        }

        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="neg_mean_squared_error",
            cv=inner_cv,
            n_jobs=None,
            refit=True
        )

        gs.fit(X_train, y_train)
        yhat[test_idx[0]] = gs.predict(X_test)[0]

        rec = {"fold": fold, "test_index": int(test_idx[0])}
        rec.update(gs.best_params_)
        rec["best_inner_score_neg_mse"] = gs.best_score_
        best_params_records.append(rec)

    r = pearsonr(y, yhat)[0]
    r2 = r2_score(y, yhat)
    best_params_df = pd.DataFrame(best_params_records)
    return yhat, r, r2, best_params_df

def nested_loocv_elastic_net_bayes(
    X,
    y,
    feature_names,
    alpha_grid=ALPHA_GRID,
    l1_ratio_grid=L1_RATIO_GRID,
    inner_cv_splits=INNER_CV_SPLITS,
    impute_strategy="mean",
    random_state=RANDOM_STATE,
):
    """
    Model 3:
      - uses same predictors as Model 2
      - fits Elastic Net on each training fold (nested tuning)
      - uses Elastic Net coefficients as prior mean for Gaussian prior
      - computes Bayesian linear-regression posterior over weights
      - predicts held-out video using posterior mean
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    loo = LeaveOneOut()
    yhat = np.full_like(y, np.nan, dtype=float)
    fold_param_records = []
    posterior_records = []

    for fold, (train_idx, test_idx) in enumerate(loo.split(X), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]

        # ---- Step 1: nested tuning for Elastic Net prior ----
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy=impute_strategy)),
            ("scaler", StandardScaler()),
            ("enet", ElasticNet(max_iter=100000, random_state=random_state))
        ])

        inner_splits = min(inner_cv_splits, len(train_idx))
        if inner_splits < 2:
            raise ValueError("Not enough training samples for inner CV.")

        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

        param_grid = {
            "enet__alpha": alpha_grid,
            "enet__l1_ratio": l1_ratio_grid
        }

        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="neg_mean_squared_error",
            cv=inner_cv,
            n_jobs=None,
            refit=True
        )
        gs.fit(X_train, y_train)

        best_alpha = gs.best_params_["enet__alpha"]
        best_l1 = gs.best_params_["enet__l1_ratio"]

        # ---- Step 2: refit on full training fold ----
        imputer = SimpleImputer(strategy=impute_strategy)
        scaler = StandardScaler()

        X_train_imp = imputer.fit_transform(X_train)
        X_test_imp = imputer.transform(X_test)

        X_train_sc = scaler.fit_transform(X_train_imp)
        X_test_sc = scaler.transform(X_test_imp)

        enet = ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=100000, random_state=random_state)
        enet.fit(X_train_sc, y_train)

        mu0 = enet.coef_.copy()
        intercept0 = float(enet.intercept_)

        resid = y_train - enet.predict(X_train_sc)
        sigma2 = float(np.mean(resid**2) + 1e-6)

        tau2 = float(np.var(mu0) + 1.0)

        # ---- Step 3: Bayesian posterior update ----
        y_center = y_train - intercept0
        XTX = X_train_sc.T @ X_train_sc
        p = X_train_sc.shape[1]
        I = np.eye(p)

        Sigma_post = np.linalg.inv((XTX / sigma2) + (I / tau2))
        mu_post = Sigma_post @ ((X_train_sc.T @ y_center) / sigma2 + (mu0 / tau2))

        yhat[test_idx[0]] = intercept0 + (X_test_sc @ mu_post)[0]

        fold_param_records.append({
            "fold": fold,
            "test_index": int(test_idx[0]),
            "enet_alpha": best_alpha,
            "enet_l1_ratio": best_l1,
            "sigma2": sigma2,
            "tau2": tau2,
            "best_inner_score_neg_mse": gs.best_score_
        })

        post_sd = np.sqrt(np.diag(Sigma_post))
        rec = {"fold": fold, "test_index": int(test_idx[0])}
        for name, m, s in zip(feature_names, mu_post, post_sd):
            rec[f"{name}__post_mean"] = m
            rec[f"{name}__post_sd"] = s
        posterior_records.append(rec)

    r = pearsonr(y, yhat)[0]
    r2 = r2_score(y, yhat)

    fold_param_df = pd.DataFrame(fold_param_records)
    posterior_df = pd.DataFrame(posterior_records)
    return yhat, r, r2, fold_param_df, posterior_df


# ============================================================
# 1) LOAD VIDEO MAPPING
# ============================================================
video_map_df = pd.read_csv(VIDEO_MAP_CSV).copy()
col_lower = {c.lower(): c for c in video_map_df.columns}

video_col = next((col_lower[c] for c in ["videokey", "video", "videoname", "basevideo"] if c in col_lower), None)
trig_col = next((col_lower[c] for c in ["trigger", "trigger_number", "trig"] if c in col_lower), None)

if video_col is None or trig_col is None:
    raise ValueError(f"video_mapping.csv must have video and trigger columns. Found: {list(video_map_df.columns)}")

video_map_df = video_map_df.rename(columns={video_col: "VideoKey", trig_col: "Trigger"})
video_map_df["VideoKey"] = video_map_df["VideoKey"].apply(norm_name)
video_map_df["Trigger"] = pd.to_numeric(video_map_df["Trigger"], errors="coerce")
video_map_df = video_map_df.dropna(subset=["VideoKey", "Trigger"]).drop_duplicates(subset=["VideoKey", "Trigger"])
video_map_df["Trigger"] = video_map_df["Trigger"].astype(int)

print("video_map_df:", video_map_df.shape)
display(video_map_df.head())


# ============================================================
# 2) LOAD VIDEO INDEX + BUILD ACROSS-VIDEO PCA FEATURES
# ============================================================
idx = pd.read_csv(EMB_INDEX_CSV).copy()
idx["video_stem"] = idx["video_stem"].apply(norm_name)
idx["VideoKey"] = idx["video_stem"]

# Keep mapped videos only
idx = idx.merge(video_map_df[["VideoKey", "Trigger"]], on="VideoKey", how="inner")

video_pc_df, scree_df, video_mean_df = build_across_video_pca_features(idx, n_video_pcs=N_VIDEO_PCS)

video_pc_cols = [f"VideoPC{i+1:02d}" for i in range(N_VIDEO_PCS)]

print("video_pc_df:", video_pc_df.shape)
display(video_pc_df.head())

SCREE_PNG = os.path.join(OUT_DIR, "video_pca_scree_plot_across_videos.png")
plot_scree(scree_df, SCREE_PNG)
print("Saved scree plot to:", SCREE_PNG)

scree_df.to_csv(os.path.join(OUT_DIR, "video_pca_explained_variance_across_videos.csv"), index=False)
video_mean_df.to_csv(os.path.join(OUT_DIR, "video_mean_embeddings_before_pca.csv"), index=False)
video_pc_df.to_csv(os.path.join(OUT_DIR, "video_pca_features_across_videos.csv"), index=False)


# ============================================================
# 3) LOAD BEHAVIOR
# ============================================================
beh = pd.read_csv(BEHAVIOR_CSV).copy()

if "VideoKey" not in beh.columns:
    if "BaseVideo" in beh.columns:
        beh["VideoKey"] = beh["BaseVideo"].apply(norm_name)
    elif "Video" in beh.columns:
        beh["VideoKey"] = beh["Video"].apply(norm_name)
    else:
        beh["VideoKey"] = beh.iloc[:, 0].apply(norm_name)

beh = beh.merge(video_map_df[["VideoKey", "Trigger"]], on="VideoKey", how="inner")

# FORCE USE OF Avg_Performance_Zscored
perf_col = find_behavior_target_col(beh)
beh[perf_col] = pd.to_numeric(beh[perf_col], errors="coerce")
print("Using performance target:", perf_col)

beh = beh[["VideoKey", "Trigger", perf_col]].drop_duplicates(subset=["VideoKey", "Trigger"])


# ============================================================
# 4) LOAD ROI ISC (WIDE) AND STANDARDIZE WITHIN EACH ROI
# ============================================================
isc_wide = pd.read_csv(ISC_CSV).copy()

if "Trigger" not in isc_wide.columns:
    trig_candidates = [c for c in isc_wide.columns if c.lower() == "trigger" or "trig" in c.lower()]
    if not trig_candidates:
        raise ValueError("ISC wide CSV must contain Trigger (or trigger-like) column.")
    isc_wide["Trigger"] = pd.to_numeric(isc_wide[trig_candidates[0]], errors="coerce")

isc_wide["Trigger"] = pd.to_numeric(isc_wide["Trigger"], errors="coerce")
isc_wide = isc_wide.dropna(subset=["Trigger"])
isc_wide["Trigger"] = isc_wide["Trigger"].astype(int)

isc_wide = isc_wide.merge(video_map_df[["Trigger", "VideoKey"]], on="Trigger", how="inner")

roi_cols = [c for c in isc_wide.columns if c.startswith("Left ") or c.startswith("Right ")]
roi_cols = sorted(roi_cols)

if len(roi_cols) != 12:
    print(f"Warning: expected 12 ROI ISC columns, found {len(roi_cols)}.")
    print("ROI columns found:", roi_cols)

# Standardize within each ROI (column-wise across videos)
for col in roi_cols:
    isc_wide[col] = zscore_series(isc_wide[col])

print("ROI ISC columns:", roi_cols)
print("isc_wide:", isc_wide.shape)
print("\nTop ROI ISC NaN counts after ROI-wise standardization:\n",
      isc_wide[roi_cols].isna().sum().sort_values(ascending=False))


# ============================================================
# 5) BUILD MASTER TABLE
# ============================================================
df = video_pc_df.merge(beh, on=["VideoKey", "Trigger"], how="inner")
df = df.merge(isc_wide[["VideoKey", "Trigger"] + roi_cols], on=["VideoKey", "Trigger"], how="inner")

print("\nMASTER DF:", df.shape)
display(df[["VideoKey", "Trigger", perf_col] + video_pc_cols[:4] + roi_cols[:4]].head())

MASTER_CSV = os.path.join(OUT_DIR, "master_model_df.csv")
df.to_csv(MASTER_CSV, index=False)
print("Saved master dataframe to:", MASTER_CSV)


# ============================================================
# 6) DEFINE MATRICES
# ============================================================
X_video = df[video_pc_cols].values
X_roi = df[roi_cols].values
X_video_roi = np.hstack([X_video, X_roi])

feature_names_24 = video_pc_cols + roi_cols
y_perf = pd.to_numeric(df[perf_col], errors="coerce").values


# ============================================================
# 7) MODEL 1: ELASTIC NET (VIDEO FEATURES ONLY)
# ============================================================
yhat_m1, r_m1, r2_m1, bp_m1 = nested_loocv_elastic_net(
    X=X_video,
    y=y_perf
)

print("\n=== MODEL 1: Elastic Net (Video features only) ===")
print(f"Target: {perf_col}")
print(f"r = {r_m1:.3f}, R^2 = {r2_m1:.3f}")

bp_m1.to_csv(os.path.join(OUT_DIR, "best_params_model1_elasticnet.csv"), index=False)


# ============================================================
# 8) MODEL 2: ELASTIC NET (VIDEO FEATURES + ROI ISC)
# ============================================================
yhat_m2, r_m2, r2_m2, bp_m2 = nested_loocv_elastic_net(
    X=X_video_roi,
    y=y_perf
)

print("\n=== MODEL 2: Elastic Net (Video features + ROI ISC) ===")
print(f"Target: {perf_col}")
print(f"r = {r_m2:.3f}, R^2 = {r2_m2:.3f}")

bp_m2.to_csv(os.path.join(OUT_DIR, "best_params_model2_elasticnet.csv"), index=False)


# ============================================================
# 9) MODEL 3: ELASTIC NET + BAYESIAN POSTERIOR
# ============================================================
yhat_m3, r_m3, r2_m3, fold_params_m3, posterior_m3 = nested_loocv_elastic_net_bayes(
    X=X_video_roi,
    y=y_perf,
    feature_names=feature_names_24
)

print("\n=== MODEL 3: Elastic Net + Bayesian posterior ===")
print(f"Target: {perf_col}")
print(f"r = {r_m3:.3f}, R^2 = {r2_m3:.3f}")

fold_params_m3.to_csv(os.path.join(OUT_DIR, "best_params_model3_elasticnet_bayes.csv"), index=False)
posterior_m3.to_csv(os.path.join(OUT_DIR, "posterior_weight_distributions_model3.csv"), index=False)


# ============================================================
# 10) SAVE PREDICTIONS + SUMMARY
# ============================================================
pred_df = df[["VideoKey", "Trigger"]].copy()
pred_df[f"y_true_{perf_col}"] = y_perf
pred_df["yhat_model1_video_elasticnet"] = yhat_m1
pred_df["yhat_model2_video_roi_elasticnet"] = yhat_m2
pred_df["yhat_model3_video_roi_elasticnet_bayes"] = yhat_m3

PRED_CSV = os.path.join(OUT_DIR, "performance_predictions_loocv.csv")
pred_df.to_csv(PRED_CSV, index=False)

summary_df = pd.DataFrame([
    {"Target": perf_col, "Model": "Model1_Video_ElasticNet", "r": r_m1, "R2": r2_m1},
    {"Target": perf_col, "Model": "Model2_Video_ROI_ElasticNet", "r": r_m2, "R2": r2_m2},
    {"Target": perf_col, "Model": "Model3_Video_ROI_ElasticNet_Bayes", "r": r_m3, "R2": r2_m3},
])

SUMMARY_CSV = os.path.join(OUT_DIR, "performance_model_comparison_summary.csv")
summary_df.to_csv(SUMMARY_CSV, index=False)

print("\n=== MODEL COMPARISON SUMMARY ===")
display(summary_df)

print("\nSaved outputs to:", OUT_DIR)
print("Predictions:", PRED_CSV)
print("Summary:", SUMMARY_CSV)